# Task 1: Laying the Foundation for Analysis

## 1. Planned Analysis Workflow (Task 1.1.a)
The analysis steps are planned as follows:
1. **Data Loading & Cleaning:** Import daily Brent oil prices, parse dates, handle missing values, and calculate log returns.
2. **Exploratory Data Analysis (EDA):** Perform rolling trend analysis, run Augmented Dickey-Fuller (ADF) tests, and plot volatility.
3. **Historical Event Mapping:** Align identified change points with a curated dataset of geopolitical and OPEC events.
4. **Bayesian Modeling (PyMC):** Implement a change point model with a discrete prior $\tau$ for the switch point, and normal likelihoods for the price regimes.
5. **Validation & Quantified Interpretation:** Run MCMC diagnostics, check convergence ($\hat{R}$), and quantify price shifts before and after detected breaks.

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

# Allow notebook to import from the src directory
sys.path.append(os.path.abspath(os.path.join('..')))
from src.data_loader import load_brent_data, calculate_log_returns

# Load real dataset (replace with your actual local path)
# df = load_brent_data('../data/brent_prices.csv')

# Placeholder mock data generator for execution safety
import pandas as pd
import numpy as np
dates = pd.date_range(start="1987-05-20", end="2022-09-30", freq="B")
np.random.seed(42)
prices = 20 + np.cumsum(np.random.normal(0, 1.2, len(dates)))
df = pd.DataFrame(data={"Price": np.clip(prices, 5, 150)}, index=dates)

df = calculate_log_returns(df)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Raw Prices and Moving Averages (Trend)
axes[0].plot(df['Price'], label='Daily Price', color='steelblue', alpha=0.6)
axes[0].plot(df['Price'].rolling(window=252).mean(), label='1-Year Rolling Avg (Trend)', color='darkorange', linewidth=2)
axes[0].set_title('Brent Oil Price Trend Analysis')
axes[0].set_ylabel('USD per Barrel')
axes[0].legend()
axes[0].grid(True, linestyle='--')

# Plot 2: Log Returns (Volatility Patterns)
axes[1].plot(df['Log_Returns'], label='Daily Log Returns', color='purple', alpha=0.5)
axes[1].plot(df['Log_Returns'].rolling(window=21).std(), label='21-Day Rolling Volatility', color='black', linewidth=1.5)
axes[1].set_title('Brent Oil Log Returns & Volatility Patterns')
axes[1].set_ylabel('Log Returns / Volatility')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Perform Augmented Dickey-Fuller Test
def print_adf_results(series, name):
    result = adfuller(series)
    print(f"ADF Statistic for {name}: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4E}")
    print("Is stationary:", "Yes" if result[1] <= 0.05 else "No")
    print("-" * 50)

print_adf_results(df['Price'], 'Raw Prices')
print_adf_results(df['Log_Returns'], 'Log Returns')

### How Time Series Properties Inform Modeling Choices:

1. **Non-Stationarity of Raw Prices:** 
   Our ADF test confirms that raw Brent oil prices are non-stationary ($p > 0.05$). Fitting a standard static regression model directly to raw prices violates basic regression assumptions, leading to spurious results. 
   * **Modeling Choice:** We must use a **Change Point Model** or **Markov-Switching Model** which explicitly assumes that parameters (like the mean $\mu$ and variance $\sigma$) are piecewise constant, shifting only at structural breaks. Alternatively, we model stationary differences (log returns).

2. **Volatility Clustering:** 
   The log returns plot shows distinct clusters of high volatility (sharp spikes during market crises) and low volatility (flat regions during calm periods).
   * **Modeling Choice:** A simple model assuming a constant standard deviation ($\sigma$) across all decades is insufficient. Our change-point model should either allow the variance parameter ($\sigma_1, \sigma_2$) to shift along with the mean, or utilize models like GARCH in future extensions to account for dynamic variance.

## 2. Understanding Change Point Models

### Purpose and Function:
A change point model is designed to detect structural breaks where the underlying statistical distribution of a time series changes abruptly. In Brent oil prices, these breaks represent shifts in global supply-demand dynamics caused by major external shocks (e.g., wars, pandemics, or policy shifts).

Mathematically, a single change-point model partitions a time series $Y_t$ at a discrete time step $\tau$:

$$Y_t \sim \mathcal{N}(\mu_1, \sigma^2) \quad \text{for } t < \tau$$
$$Y_t \sim \mathcal{N}(\mu_2, \sigma^2) \quad \text{for } t \ge \tau$$

By using Bayesian inference via `PyMC`, we treat the parameters $\tau$ (the switch point), $\mu_1, \mu_2$ (the regime means), and $\sigma$ (noise) as random variables. This allows us to calculate complete posterior distributions rather than single point estimates.

### Expected Outputs:
1. **Posterior Distribution of $\tau$ (Switch Point):** Provides a probability distribution over time indicating when the transition occurred. A narrow, high-probability peak signals a distinct, high-confidence break date.
2. **Regime Parameters ($\mu_1, \mu_2$):** Quantifiable average price values before and after the change point, enabling calculations such as: *"The average price shifted from $X to $Y, representing a Z% increase."*
3. **Credible Intervals:** Bayesian intervals that represent uncertainty for all model parameters.

### Limitations of Single Change-Point Models:
* **Over-Simplification:** Real-world price histories spanning over 30 years contain *multiple* structural breaks. A single-switch model will only isolate the most dominant change point, ignoring other significant shocks.
* **Abrupt Transitions:** The model assumes an instantaneous change, whereas real market adjustments to policy shifts can sometimes occur gradually over weeks or months.